## Real-time Wearable Data Ingestion via WebSockets with Encryption

This notebook demonstrates the core components required to build an API endpoint for real-time wearable data (heart rate/steps) ingestion using WebSockets, with encryption before database insertion.

**Key Components:**
1.  **WebSocket Server**: To handle real-time, bi-directional communication.
2.  **Encryption**: To secure data payloads before storage.
3.  **Database Interaction**: To persist the encrypted data.

**Note on Colab Environment:**
Directly running a persistent WebSocket server or a production database within a Google Colab notebook is not feasible for a continuous service. However, we can simulate the logic and demonstrate the core functionalities (like data reception, encryption, and conceptual database interaction) within the notebook environment.

### 1. Install Necessary Libraries
We'll need `websockets` for WebSocket communication and `cryptography` for encryption.

In [1]:
%%capture
!pip install websockets cryptography

### 2. Encryption Setup
We'll use `cryptography`'s `Fernet` symmetric encryption. First, let's generate a key that will be used for both encryption and decryption. In a real-world scenario, this key would be securely managed and not hardcoded or exposed.

In [2]:
from cryptography.fernet import Fernet

# Generate a key and instantiate a Fernet object
key = Fernet.generate_key()
fernet = Fernet(key)

print(f"Encryption Key Generated: {key.decode()}")
print("Note: In a production environment, this key would be securely stored and managed, not printed or hardcoded.")

def encrypt_data(data: str) -> bytes:
    """Encrypts a string using the generated Fernet key."""
    return fernet.encrypt(data.encode())

def decrypt_data(encrypted_data: bytes) -> str:
    """Decrypts bytes using the generated Fernet key."""
    return fernet.decrypt(encrypted_data).decode()

# Example Usage (for demonstration)
original_data = "{'heart_rate': 72, 'steps': 1200}"
encrypted = encrypt_data(original_data)
decrypted = decrypt_data(encrypted)

print(f"\nOriginal: {original_data}")
print(f"Encrypted: {encrypted}")
print(f"Decrypted: {decrypted}")

assert original_data == decrypted
print("Encryption/Decryption successful!")

Encryption Key Generated: Q6n0LhyK_k6zekPFEuOc2x7OjhFUsj6S2bEg-yqCVcU=
Note: In a production environment, this key would be securely stored and managed, not printed or hardcoded.

Original: {'heart_rate': 72, 'steps': 1200}
Encrypted: b'gAAAAABp1z-PrCT85gV0hmXfg0oRNdIuk-2VQR7CmMokMtmJeg8c226c9ZbBVOmCip9HQZw2YJxDQlxhCDGisODJ0qsru6cyVN3aOJBumaponMhVmkrHP41mFPDNeZaa5Pp6diiMDpm1'
Decrypted: {'heart_rate': 72, 'steps': 1200}
Encryption/Decryption successful!


### 3. WebSocket Server Simulation

In a real-world scenario, you would have a dedicated server application running a WebSocket server. Here, we'll simulate the server's core logic within Colab by defining a `websocket_handler` function that would typically run on the server. This handler will receive data, encrypt it, and then conceptually log or store it.

Since Colab notebooks have limitations with persistent network services and background processes, directly running a `websockets.serve` function continuously is not ideal for demonstration. Instead, we will focus on the `websocket_handler` function and then show how a client would interact with it conceptually.

In [3]:
import asyncio
import json
from websockets.server import serve

async def websocket_handler(websocket):
    try:
        async for message in websocket:
            print(f"\nReceived raw data: {message}")

            # Simulate data parsing and validation (e.g., JSON)
            try:
                data = json.loads(message)
                # Example: data validation based on expected fields
                if not all(k in data for k in ['heart_rate', 'steps']):
                    raise ValueError("Missing expected fields: 'heart_rate' or 'steps'")
            except json.JSONDecodeError:
                print("Error: Received non-JSON data.")
                await websocket.send("Error: Invalid JSON format.")
                continue
            except ValueError as e:
                print(f"Error: {e}")
                await websocket.send(f"Error: {e}")
                continue

            # Encrypt the received data
            encrypted_payload = encrypt_data(message) # Encrypt the raw string message
            print(f"Encrypted payload: {encrypted_payload}")

            # --- Conceptual Database Storage ---
            # In a real application, you would now store `encrypted_payload`
            # into a database along with a timestamp or user ID.
            # For demonstration, we'll just print a message.
            print("Conceptual: Encrypted data ready for database insertion.")

            # Simulate a successful response to the client
            response = {"status": "success", "message": "Data received and encrypted.", "encrypted_size": len(encrypted_payload)}
            await websocket.send(json.dumps(response))

    except Exception as e:
        print(f"WebSocket handler error: {e}")

print("WebSocket handler defined. This function would process incoming messages on a running server.")


WebSocket handler defined. This function would process incoming messages on a running server.


/tmp/ipykernel_4607/3425106721.py:3: DeprecationWarning: websockets.server.serve is deprecated
  from websockets.server import serve


### 4. WebSocket Client Simulation

To complete our conceptual demonstration, we'll simulate a client connecting to a WebSocket server and sending data. In a real-world application, this client could be a wearable device, a mobile app, or another server.

Since we can't run `websockets.serve` persistently in Colab, we will simulate the entire server-client interaction within a single asynchronous block. This will involve:
1.  Starting a minimal WebSocket server (which will immediately exit after handling a connection).
2.  A client connecting to this server.
3.  The client sending data.
4.  The server (our `websocket_handler`) processing and encrypting the data.
5.  The server sending a response back to the client.

In [6]:
import asyncio
import json
from websockets.server import serve
from websockets.client import connect

async def simulate_websocket_interaction():
    # This function will run a temporary server for a single interaction
    # and then close, allowing the client to connect and send data.
    async def server_main(websocket, path):
        await websocket_handler(websocket)

    # Start a temporary WebSocket server on a local port
    # In a real scenario, this would be a long-running external server
    # Using a different port to avoid 'address already in use' errors
    server_task = await serve(server_main, "localhost", 8766)
    print("\n--- Simulating Server Start ---")

    try:
        # Simulate a client connecting and sending data
        uri = "ws://localhost:8766"
        async with connect(uri) as websocket:
            print("--- Client Connected ---")

            # Client sends wearable data
            client_data = "{\"heart_rate\": 75, \"steps\": 1500, \"timestamp\": \"2023-10-27T10:00:00Z\"}"
            print(f"Client sending: {client_data}")
            await websocket.send(client_data)

            # Client waits for a response from the server
            response = await websocket.recv()
            print(f"Client received response: {response}")

            parsed_response = json.loads(response)
            if parsed_response.get("status") == "success":
                print("Client confirms data was received and encrypted by server.")

    except Exception as e:
        print(f"Simulation error: {e}")
    finally:
        # Correct way to close the server task
        server_task.close()
        await server_task.wait_closed()
        print("--- Simulating Server Stop ---")

print("Initiating WebSocket client-server interaction simulation...")
await simulate_websocket_interaction()


Initiating WebSocket client-server interaction simulation...

--- Simulating Server Start ---
--- Client Connected ---
Client sending: {"heart_rate": 75, "steps": 1500, "timestamp": "2023-10-27T10:00:00Z"}

Received raw data: {"heart_rate": 75, "steps": 1500, "timestamp": "2023-10-27T10:00:00Z"}
Encrypted payload: b'gAAAAABp10ABygdIDeXuL3DJYXrK9NfOrpijb1jP_97w5HQyQbuqNU8UstSkK_4QZDlDbexbgmU_tiIQ1KKT29me6q8_XWSm8guGrokizoZna_FsT6aZaAzL_elU513cINwmrtsn4fzmQAgx8Fhm9Q-4X3B09PKyn5g9HX9glZL8lS7hMwpMwQc='
Conceptual: Encrypted data ready for database insertion.
Client received response: {"status": "success", "message": "Data received and encrypted.", "encrypted_size": 184}
Client confirms data was received and encrypted by server.
--- Simulating Server Stop ---


/tmp/ipykernel_4607/3600705853.py:3: DeprecationWarning: websockets.server.serve is deprecated
  from websockets.server import serve
/tmp/ipykernel_4607/3600705853.py:4: DeprecationWarning: websockets.client.connect is deprecated
  from websockets.client import connect
